In [0]:
%sql

-- Fraud and Short-Burst Pattern Analysis: by Amount Bucket and KYC Status

SELECT 
    f.amount_bucket,
    COALESCE(c.KYC_Status,'UNKNOWN') AS KYC_Status,
    COUNT(*) AS total_transactions,
    SUM(f.amount) AS total_transaction_volume,
    ROUND(AVG(COALESCE(c.Risk_Score,0)),2) AS avg_fraud_risk_score,

    CASE
        WHEN AVG(c.Risk_Score) >= 80 
             AND COALESCE(c.KYC_Status,'UNKNOWN') = 'PENDING'
        THEN 'HIGH RISK'

        WHEN AVG(c.Risk_Score) >= 50
        THEN 'MEDIUM RISK'

        ELSE 'LOW RISK'
    END AS risk_category

FROM banking_gold.fact_wide_transactions f

LEFT JOIN banking_gold.dim_customer_gold_scd1 c 
ON f.CUST_ID = c.CUST_ID

WHERE f.transaction_status = 'SUCCESS'

AND f.transaction_hour IN (
    SELECT transaction_hour
    FROM banking_gold.fact_wide_transactions
    GROUP BY transaction_hour
    HAVING COUNT(*) >= 2
)

GROUP BY 
    f.amount_bucket,
    COALESCE(c.KYC_Status,'UNKNOWN')

ORDER BY total_transaction_volume DESC;

In [0]:
%sql
-- Balance Breach & Account Takeover Analysis: by Account Type and Status
SELECT 
    COALESCE(a.TYPE, 'UNKNOWN') AS account_type,
    COALESCE(a.STATUS, 'UNKNOWN') AS account_status,

    COUNT(*) AS breach_attempts,
    SUM(f.amount) AS total_attempted_fraud_volume,
    ROUND(AVG(f.BALANCE), 2) AS avg_account_balance

FROM banking_gold.fact_wide_transactions f

LEFT JOIN banking_gold.dim_account_gold_scd1 a 
    ON f.ACCOUNT_ID = a.ACCOUNT_ID

WHERE f.transaction_status = 'FAILED' 
  AND f.amount_bucket = 'MEDIUM'

GROUP BY 
    COALESCE(a.TYPE, 'UNKNOWN'),
    COALESCE(a.STATUS, 'UNKNOWN')

ORDER BY total_attempted_fraud_volume DESC;

In [0]:
%sql
-- Money Laundering & Smurfing Risk Analysis: by KYC Status

SELECT 
    COALESCE(c.KYC_Status, 'UNKNOWN') AS KYC_Status,

    ROUND(AVG(c.Risk_Score), 2) AS avg_risk_score,

    COUNT(*) AS smurfing_frequency,
    SUM(f.amount) AS total_laundered_volume,
    ROUND(AVG(f.amount), 2) AS avg_smurfing_transaction_value

FROM banking_gold.fact_wide_transactions f

LEFT JOIN banking_gold.dim_customer_gold_scd1 c 
ON f.CUST_ID = c.CUST_ID

WHERE UPPER(c.KYC_Status) = 'PENDING'

GROUP BY COALESCE(c.KYC_Status, 'UNKNOWN')

ORDER BY total_laundered_volume DESC;

In [0]:
%sql
-- Dormant Account Reactivation Fraud Analysis (Improved Version)

SELECT 
    a.TYPE AS account_type,
    f.amount_bucket,

    COUNT(*) AS drainage_calls,
    SUM(f.amount) AS total_drained_volume,
    ROUND(AVG(c.Risk_Score), 2) AS avg_compromised_risk_score

FROM banking_gold.fact_wide_transactions f
LEFT JOIN banking_gold.dim_account_gold_scd1 a 
    ON f.ACCOUNT_ID = a.ACCOUNT_ID
LEFT JOIN banking_gold.dim_customer_gold_scd1 c 
    ON f.CUST_ID = c.CUST_ID

WHERE a.STATUS IN ('ACTIVE', 'DORMANT')  
  AND c.Risk_Score > 80
  AND f.transaction_status = 'SUCCESS'
  AND f.amount_bucket IN ('LOW','MEDIUM', 'HIGH')

GROUP BY 
    a.TYPE, 
    f.amount_bucket

ORDER BY total_drained_volume DESC;


In [0]:
%sql
--Fraud Operations High-Risk Aggregations: by Transaction Date and Hour

CREATE OR REPLACE VIEW banking_gold.bi_realtime_telecom_style_fraud_dashboard AS
SELECT 
    f.transaction_date,
    f.transaction_hour,
    COUNT(*) AS total_risk_transactions,
    SUM(f.amount) AS total_volume_at_risk,
    ROUND(AVG(c.Risk_Score), 2) AS average_customer_risk_index
FROM banking_gold.fact_wide_transactions f
LEFT JOIN banking_gold.dim_customer_gold_scd1 c ON f.CUST_ID = c.CUST_ID
WHERE c.Risk_Score > 80 OR f.transaction_status = 'FAILED'
GROUP BY f.transaction_date, f.transaction_hour;



In [0]:
%sql
SELECT * 
FROM banking_gold.bi_realtime_telecom_style_fraud_dashboard;